# SVM
## Training an SVM classifier on top of the extracted CNN Nodes 

In [ ]:
# generic
import numpy as np
import os
import matplotlib.pyplot as plt
import h5py
import itertools
import matplotlib
#matplotlib.style.use('ggplot')

# sklearn
from sklearn import cross_validation, grid_search
from sklearn.metrics import confusion_matrix, classification_report,accuracy_score
from sklearn.svm import SVC, LinearSVC
from sklearn.model_selection import GridSearchCV
import pandas as pd


importing necessary data 

In [ ]:
X_test = np.load("data/SVM-X_test.npy")
y_test = pd.read_csv("data/SVM-y_test.csv")
y_test = y_test['0']

In [ ]:
X_train = np.load("data/SVM-X_train.npy")
y_train = pd.read_csv("data/SVM-y_train.csv")
y_train = list(y_train['0'])

In [ ]:
print(len(X_test), len(y_test))
print(len(X_train), len(y_train))

In [ ]:
gt = pd.read_csv(os.path.join('data/sc5-test','ground_truth.txt'),sep = ';', header = None)
y_test_new = []
for ind, val in enumerate(y_test):
    y_test_new.append(list(gt[gt[0]==val+'.jpg'][1])[0])

In [ ]:
from numpy import array
from numpy import argmax
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA

def onehot(y_test_new):
    values = np.array(y_test_new)
    print(values)
    # integer encode
    label_encoder = LabelEncoder()
    integer_encoded = label_encoder.fit_transform(values)
    # binary encode
    onehot_encoder = OneHotEncoder(sparse=False)
    integer_encoded = integer_encoded.reshape(len(integer_encoded), 1)
    onehot_encoded = onehot_encoder.fit_transform(integer_encoded)
    return integer_encoded
y_test = onehot(y_test_new)
y_train = onehot(y_train)

In [ ]:
def scale_set(train,test):
    """uses sklearn standard sclar to normalize data"""
    sc = StandardScaler()
    fitted = sc.fit(train)
    return sc.transform(train), sc.transform(test)
#X_train, X_test = scale_set(X_train, X_test)
def my_PCA(train,test):
    '''applies PCA to the data'''
    pca = PCA(n_components=train.shape[1])
    data = pca.fit(train)
    return pca.transform(train), pca.transform(test)
X_train, X_test = my_PCA(X_train, X_test)

In [ ]:
y_test = y_test.ravel()
y_train = y_train.ravel()

# Now I will train three SVM classifiers:
* SGD
* Linear SVC
* SVC (and with bagging)

## SGD

In [ ]:
from sklearn.linear_model import SGDClassifier
np.random.seed(42)

In [ ]:
clf = SGDClassifier(loss="hinge",
                    penalty="l2",
                    n_jobs=-1,
                    learning_rate='invscaling',
                    eta0 = 3,
                    n_iter = 112)
clf.fit(X_train, y_train)

In [ ]:
y_predict=clf.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, y_predict))
print("The accuracy is {:.1f}%".format(accuracy_score(y_test, y_predict, normalize=True)*100))

## LinearSVC

In [ ]:
clf = LinearSVC(random_state=42,C=100,dual=False)
clf.fit(X_train, y_train)

In [ ]:
y_predict=clf.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, y_predict))
print("The accuracy is {:.1f}%".format(accuracy_score(y_test, y_predict, normalize=True)*100))

## SVC (kernel trick) 
  

In [ ]:
clf = SVC(decision_function_shape='ovr')
clf.fit(X_train, y_train)

In [ ]:
print("\nBest parameters set:")
print(clf.best_params_)

In [ ]:
y_predict=clf.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, y_predict))
print("The accuracy is {:.1f}%".format(accuracy_score(y_test, y_predict, normalize=True)*100))